# Unsloth Qwen3-1.7B SFT / Batch Inference Notebook

这个 notebook 针对你的实验数据格式优化：train 和 test 已经拆成两个独立 `.json` 文件；每个文件是 JSON array。原始样本可以是你现在的字段结构，例如：

```json
{
  "prompt_id": "claim001_highInv_highExpert_StrongArg",
  "prompt": "...",
  "groundtruth": "My attitude score toward this proposal is: 8",
  "condition": {"involvement": "high", "source_expertise": "high", "argument_quality": "strong"}
}
```

Notebook 会先把原始 train/test 转成标准 chat SFT 数据：

```json
{
  "messages": [
    {"role": "system", "content": "You are a respondent in a persuasion scenario. Answer from the assigned role's perspective."},
    {"role": "user", "content": "这里放 prompt"},
    {"role": "assistant", "content": "这里放 groundtruth"}
  ]
}
```

注意：很多 Qwen / Unsloth chat template 只接受 `system` / `user` / `assistant`，所以这里最后一轮用 `assistant` role；“respondent” 身份放在 system prompt 里表达。

Qwen3 的 chat template 可能默认插入 `<think>...</think>`。这个 notebook 默认通过 `ENABLE_THINKING = False` 关闭 thinking，避免 SFT 文本里多出空 thinking 标签。

Notebook 流程保持简洁：

1. 选择 GPU。
2. 加载本地 `model/Qwen3-1.7B`。
3. 将 `TRAIN_FILE` / `TEST_FILE` 标准化成 messages 数据。
4. 用标准化后的 train messages 做 SFT。
5. 用标准化后的 test messages 批量推理，保留原始元数据并写出 `model_output` 和 `parsed_score`。
6. 可选：GRPO/RL。


## 0. 指定 GPU（最重要）

Unsloth / PyTorch 会把 `CUDA_VISIBLE_DEVICES` 看到的第一个 GPU 当作 `cuda:0`。

推荐方式是在启动 Jupyter 前指定物理 GPU：

```bash
CUDA_VISIBLE_DEVICES=1 jupyter notebook notebooks/unsloth_sft_grpo_qwen3.ipynb
```

如果没有 `jupyter` 命令，也可以用：

```bash
CUDA_VISIBLE_DEVICES=1 python -m notebook notebooks/unsloth_sft_grpo_qwen3.ipynb
```

如果已经打开 notebook，也可以在下面 cell 里设置；但必须保证这是 kernel 启动后第一个执行的 Python cell，且之前没有 import 过 `torch` / `unsloth`。如果你已经运行过后面的 cell，请 **Restart Kernel** 后再改这里。

In [ ]:
import os

# 改这里指定物理 GPU，例如 "0"、"1"、"2"。
# 设置后，notebook 内部看到的 GPU 会重新编号为 cuda:0。
SELECTED_GPU = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = SELECTED_GPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# 减少显存碎片导致的 OOM；必须在 import torch 前设置。
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

## 1. 实验配置

你已经下载好的模型默认放在仓库根目录：`model/Qwen3-1.7B`。

In [ ]:
from pathlib import Path
import sys

# 自动识别仓库根目录：既支持从 repo root 打开，也支持从 notebooks/ 打开。
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

# ========== 你通常只需要改这里 ==========
MODEL_NAME_OR_PATH = str(REPO_ROOT / "model" / "Qwen3-1.7B")
TRAIN_FILE = REPO_ROOT / "data" / "train.json"  # 改成你的原始训练集 JSON 文件
TEST_FILE = REPO_ROOT / "data" / "test.json"    # 改成你的原始测试集 JSON 文件
PROCESSED_TRAIN_FILE = REPO_ROOT / "outputs" / "processed_train_messages.json"
PROCESSED_TEST_FILE = REPO_ROOT / "outputs" / "processed_test_messages.json"
SFT_OUTPUT_DIR = REPO_ROOT / "outputs" / "qwen3_1p7b_unsloth_lora"
PRED_OUTPUT_FILE = REPO_ROOT / "outputs" / "qwen3_1p7b_test_predictions.json"

# 你的原始数据字段：如果答案字段不叫 groundtruth，把候选字段加到 RESPONSE_FIELD_CANDIDATES 里。
PROMPT_FIELD = "prompt"
RESPONSE_FIELD = "groundtruth"
RESPONSE_FIELD_CANDIDATES = ["groundtruth", "response", "answer", "label", "output", "completion"]
ID_FIELD = "prompt_id"
SYSTEM_PROMPT = "You are a respondent in a persuasion scenario. Answer from the assigned role's perspective."
ASSISTANT_ROLE = "assistant"  # 不建议改成 respondent；多数 chat template 只支持 assistant。
ENABLE_THINKING = False      # Qwen3 默认可能插入 <think>...</think>；SFT/量表任务建议关闭。

# 模型 / QLoRA 设置
MAX_SEQ_LENGTH = 1024       # 如果 OOM，先降到 512
LOAD_IN_4BIT = True         # True=QLoRA；False=普通 LoRA
DTYPE = None                # None 让 Unsloth 自动选择

# LoRA 超参
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# SFT 超参
RESPONSE_ONLY_LOSS = True   # True=只训练 groundtruth/assistant answer；False=prompt+answer 都算 loss
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4

# 批量推理参数
MAX_NEW_TOKENS = 32         # 你的任务只需要输出 Likert 句子，32/64 通常足够
TEMPERATURE = 0.0           # 量表预测建议先用 0，稳定可复现
TOP_P = 1.0
INFER_BATCH_SIZE = 1        # 3090 上先用 1；显存充足再调大
NUM_REPEATS = 1
MIN_FREE_GPU_MEMORY_GB = 6.0

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_NAME_OR_PATH)
print("Train file:", TRAIN_FILE)
print("Test file:", TEST_FILE)
print("Processed train:", PROCESSED_TRAIN_FILE)
print("Processed test:", PROCESSED_TEST_FILE)
print("Model exists:", Path(MODEL_NAME_OR_PATH).exists())
print("Train exists:", TRAIN_FILE.exists())
print("Test exists:", TEST_FILE.exists())


## 2. 导入依赖并检查 CUDA

In [ ]:
import json
import re
from typing import Any

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

from llm_lab.data import _apply_chat_template, load_sft_dataset, load_grpo_dataset
from llm_lab.model_utils import ensure_pad_token, print_cuda_info, require_min_cuda_memory
from llm_lab.train_utils import ResponseOnlyDataCollator, build_sft_trainer, get_training_args

print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print_cuda_info()
require_min_cuda_memory(MIN_FREE_GPU_MEMORY_GB, context="notebook Unsloth run")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用。请检查驱动、PyTorch CUDA wheel、CUDA_VISIBLE_DEVICES。")
if not Path(MODEL_NAME_OR_PATH).exists():
    raise FileNotFoundError(f"未找到模型目录：{MODEL_NAME_OR_PATH}")
if not TRAIN_FILE.exists():
    raise FileNotFoundError(f"未找到训练集文件：{TRAIN_FILE}")
if not TEST_FILE.exists():
    print(f"Warning: 未找到测试集文件：{TEST_FILE}；后面的 batch inference 会跳过，除非你修改 TEST_FILE。")
print("Current CUDA device:", torch.cuda.current_device(), torch.cuda.get_device_name(0))


### 如果这里提示显存不足怎么办？

如果报错类似 `GPU 0 only has 0.30 GiB free`，说明 notebook 当前可见的 GPU 已经被占用。请不要继续调 batch size，先处理 GPU：

1. 在终端运行 `nvidia-smi` 看哪张物理卡空闲。
2. 重启 kernel，并在第 0 个 cell 把 `SELECTED_GPU` 改成空闲卡号；或用 `CUDA_VISIBLE_DEVICES=空闲卡号 jupyter notebook ...` 启动。
3. 确认第 2 节打印的 free memory 足够后再加载模型。

当前 notebook 已把 `MAX_SEQ_LENGTH=512`、`INFER_BATCH_SIZE=1` 作为保守默认值，方便先跑通。

## 3. 加载模型并注入 LoRA

这部分基本等同于 Unsloth 官方文档里的核心代码。

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME_OR_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

ensure_pad_token(tokenizer)
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. 读取并标准化 train/test 两个 JSON 文件

这里不再依赖 `split` 字段。Notebook 会把原始字段数据转成标准 `messages` 格式：

- train：`system + user(prompt) + assistant(groundtruth)`，用于 SFT。
- test：`system + user(prompt)`；如果 test 里也有 `groundtruth`，会保留下来用于评估。

如果你的答案字段不叫 `groundtruth`，优先在配置区修改 `RESPONSE_FIELD_CANDIDATES`。


In [ ]:
def read_json_or_jsonl(path: str | Path) -> list[dict[str, Any]]:
    path = Path(path)
    raw = path.read_text(encoding="utf-8").strip()
    if not raw:
        raise ValueError(f"Empty data file: {path}")
    if path.suffix.lower() == ".json" or raw.startswith("["):
        rows = json.loads(raw)
    else:
        rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
        raise ValueError("Data must be a JSON array or JSONL of objects.")
    return rows


def first_text_field(row: dict[str, Any], candidates: list[str]) -> tuple[str | None, str | None]:
    for field in candidates:
        value = row.get(field)
        if isinstance(value, str) and value.strip():
            return field, value.strip()
    return None, None


def normalize_role(role: str) -> str:
    role = role.strip().lower()
    if role in {"respondent", "respondent ", "assistant"}:
        return "assistant"
    if role in {"system", "user"}:
        return role
    return role


def normalize_messages(messages: list[dict[str, Any]], require_response: bool) -> tuple[list[dict[str, str]], str | None]:
    normalized: list[dict[str, str]] = []
    for message in messages:
        role = normalize_role(str(message.get("role", "")))
        content = message.get("content")
        if role and isinstance(content, str) and content.strip():
            normalized.append({"role": role, "content": content.strip()})
    if not normalized:
        raise ValueError("messages is empty or malformed")
    response = normalized[-1]["content"] if normalized[-1]["role"] == "assistant" else None
    if require_response and response is None:
        raise ValueError("train messages must end with an assistant/respondent answer")
    return normalized, response


def row_to_standard_messages(row: dict[str, Any], require_response: bool) -> dict[str, Any]:
    # 保留原始元数据，额外写入标准 messages；训练统一使用 messages。
    out = dict(row)

    messages = row.get("messages")
    if isinstance(messages, list) and messages:
        normalized, response = normalize_messages(messages, require_response=require_response)
    else:
        prompt = row.get(PROMPT_FIELD)
        if not isinstance(prompt, str) or not prompt.strip():
            raise ValueError(f"missing non-empty prompt field {PROMPT_FIELD!r}")
        response_field, response = first_text_field(row, RESPONSE_FIELD_CANDIDATES)
        if require_response and response is None:
            raise ValueError(
                "missing answer field; tried "
                f"{RESPONSE_FIELD_CANDIDATES}. Add your real answer field to RESPONSE_FIELD_CANDIDATES."
            )
        normalized = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt.strip()},
        ]
        if response is not None:
            normalized.append({"role": ASSISTANT_ROLE, "content": response})
            out["response_source_field"] = response_field

    out["messages"] = normalized
    if response is not None:
        out[RESPONSE_FIELD] = response  # 统一后续评估字段名。
    return out


def standardize_rows(rows: list[dict[str, Any]], name: str, require_response: bool) -> list[dict[str, Any]]:
    if not rows:
        raise ValueError(f"{name} is empty")
    converted = []
    for idx, row in enumerate(rows):
        try:
            converted.append(row_to_standard_messages(row, require_response=require_response))
        except ValueError as exc:
            raise ValueError(f"{name} row {idx} cannot be converted to messages: {exc}") from exc
    return converted


train_raw_rows = read_json_or_jsonl(TRAIN_FILE)
test_raw_rows = read_json_or_jsonl(TEST_FILE) if TEST_FILE.exists() else []
train_rows = standardize_rows(train_raw_rows, "train", require_response=True)
test_rows = standardize_rows(test_raw_rows, "test", require_response=False) if test_raw_rows else []

PROCESSED_TRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_TRAIN_FILE.write_text(json.dumps(train_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
PROCESSED_TEST_FILE.write_text(json.dumps(test_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print(f"Train rows: {len(train_rows)} from {TRAIN_FILE}")
print(f"Test rows : {len(test_rows)} from {TEST_FILE}")
print(f"Wrote processed train messages to: {PROCESSED_TRAIN_FILE}")
print(f"Wrote processed test messages to : {PROCESSED_TEST_FILE}")
print("Train columns:", sorted(train_rows[0].keys()))
if test_rows:
    print("Test columns:", sorted(test_rows[0].keys()))
print("Example id:", train_rows[0].get(ID_FIELD))
print("Example messages:")
print(json.dumps(train_rows[0]["messages"], ensure_ascii=False, indent=2)[:1200])


## 5. 构造 SFT Dataset

这里训练不直接吃原始 JSON，而是吃上一步写出的 `PROCESSED_TRAIN_FILE`：

- 每条样本都有标准 `messages`。
- `system` 说明模型要作为 respondent 回答。
- `user` 放原始 `prompt`。
- `assistant` 放标准化后的答案字段（默认优先 `groundtruth`，也支持 `response` / `answer` / `label` 等候选）。
- `RESPONSE_ONLY_LOSS=True` 时，loss 只算 assistant answer 部分，不会训练 prompt。


In [ ]:
train_dataset = load_sft_dataset(
    PROCESSED_TRAIN_FILE,
    tokenizer,
    response_only_loss = RESPONSE_ONLY_LOSS,
    enable_thinking = ENABLE_THINKING,
)

print(train_dataset)
print("Columns:", train_dataset.column_names)
print("First row keys:", train_dataset[0].keys())


## 6. SFT 训练

- `RESPONSE_ONLY_LOSS=True`：使用 `Trainer + ResponseOnlyDataCollator`，只对 `groundtruth` 计算 loss。
- `RESPONSE_ONLY_LOSS=False`：使用 TRL `SFTTrainer`，对完整 prompt+answer 计算 loss。

In [ ]:
from transformers import Trainer

fp16 = not is_bfloat16_supported()
bf16 = is_bfloat16_supported()

training_args = get_training_args(
    output_dir = str(SFT_OUTPUT_DIR),
    max_length = MAX_SEQ_LENGTH,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    fp16 = fp16,
    bf16 = bf16,
)

if RESPONSE_ONLY_LOSS:
    trainer = Trainer(
        model = model,
        args = training_args,
        train_dataset = train_dataset,
        data_collator = ResponseOnlyDataCollator(tokenizer, max_length=MAX_SEQ_LENGTH),
    )
else:
    trainer = build_sft_trainer(model, tokenizer, train_dataset, training_args)

trainer.train()

## 7. 保存 LoRA adapter / 合并模型

In [ ]:
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(str(SFT_OUTPUT_DIR))
tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))
print("Saved LoRA adapter to:", SFT_OUTPUT_DIR)

# 如果需要部署合并模型，取消下面一行注释：
# trainer.model.save_pretrained_merged(str(SFT_OUTPUT_DIR) + "_merged_16bit", tokenizer, save_method="merged_16bit")

## 8. 单条推理 sanity check

从测试集拿一条 prompt 看模型输出格式是否正确。

In [ ]:
def generation_prompt_from_row(row: dict[str, Any]) -> str:
    messages = row.get("messages")
    if isinstance(messages, list) and messages:
        prompt_messages = [dict(message) for message in messages]
        if prompt_messages and str(prompt_messages[-1].get("role", "")).strip().lower() in {"assistant", "respondent"}:
            prompt_messages = prompt_messages[:-1]
    else:
        prompt_messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row[PROMPT_FIELD]},
        ]
    return _apply_chat_template(
        tokenizer,
        prompt_messages,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )


def generate_one(row: dict[str, Any]) -> str:
    text = generation_prompt_from_row(row)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    do_sample = TEMPERATURE > 0
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": do_sample,
        "use_cache": True,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs.update({"temperature": TEMPERATURE, "top_p": TOP_P})
    outputs = trainer.model.generate(**inputs, **generation_kwargs)
    new_tokens = outputs[:, inputs.input_ids.shape[-1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

sample = test_rows[0] if test_rows else train_rows[0]
print("Prompt id:", sample.get(ID_FIELD))
print("Groundtruth:", sample.get(RESPONSE_FIELD))
print("Model output:", generate_one(sample))


## 9. 测试集批量推理并保存结果

输出文件会保留你的原始字段（如 `prompt_id`、`claim`、`condition`、`groundtruth` 等），并新增：

- `model_output`：模型完整输出。
- `parsed_score`：从输出中抽取的 1-11 分数；解析失败则为 `None`。
- `is_exact_match`：模型输出和 `groundtruth` 去空格后是否完全一致。

In [ ]:
def parse_likert_score(text: str) -> int | None:
    # 优先匹配任务指定前缀后的数字；失败再退化为最后一个 1-11 数字。
    prefix_match = re.search(r"My attitude score toward this proposal is:\s*(\d{1,2})", text, flags=re.I)
    candidates = [prefix_match.group(1)] if prefix_match else re.findall(r"\b(?:1[01]|[1-9])\b", text)
    if not candidates:
        return None
    score = int(candidates[-1])
    return score if 1 <= score <= 11 else None


def batch_generate(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if not rows:
        return []
    results = [dict(row) for row in rows]
    pending: list[tuple[int, str]] = []
    for row_idx, row in enumerate(rows):
        text = generation_prompt_from_row(row)
        for _ in range(NUM_REPEATS):
            pending.append((row_idx, text))

    outputs_by_row = [[] for _ in rows]
    do_sample = TEMPERATURE > 0
    for start in range(0, len(pending), INFER_BATCH_SIZE):
        batch = pending[start:start + INFER_BATCH_SIZE]
        row_ids = [row_id for row_id, _ in batch]
        texts = [text for _, text in batch]
        inputs = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
        generation_kwargs = {
            "max_new_tokens": MAX_NEW_TOKENS,
            "do_sample": do_sample,
            "use_cache": True,
            "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        }
        if do_sample:
            generation_kwargs.update({"temperature": TEMPERATURE, "top_p": TOP_P})
        outputs = trainer.model.generate(**inputs, **generation_kwargs)
        new_tokens = outputs[:, inputs.input_ids.shape[-1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        for row_id, output_text in zip(row_ids, decoded):
            outputs_by_row[row_id].append(output_text.strip())
        print(f"Processed {min(start + len(batch), len(pending))}/{len(pending)} generations")

    for row, outs in zip(results, outputs_by_row):
        output_value = outs if NUM_REPEATS > 1 else outs[0]
        first_output = outs[0] if outs else ""
        row["model_output"] = output_value
        row["parsed_score"] = parse_likert_score(first_output)
        row["is_exact_match"] = first_output.strip() == str(row.get(RESPONSE_FIELD, "")).strip()
    return results


pred_rows = batch_generate(test_rows)
PRED_OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
PRED_OUTPUT_FILE.write_text(json.dumps(pred_rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(f"Wrote {len(pred_rows)} predictions to {PRED_OUTPUT_FILE}")
if pred_rows:
    print(json.dumps({k: pred_rows[0].get(k) for k in [ID_FIELD, RESPONSE_FIELD, "model_output", "parsed_score", "is_exact_match"]}, ensure_ascii=False, indent=2))


## 10. 可选：GRPO / RL

默认不运行。你要做 RL 时再取消注释，并把 reward 函数替换成你的任务 reward。例如你的任务可以用 `parsed_score` 与目标分数的距离设计 reward。

```python
# from unsloth import PatchFastRL
# PatchFastRL("grpo", FastLanguageModel)
# from trl import GRPOConfig, GRPOTrainer
# rl_dataset = load_grpo_dataset(PROCESSED_TRAIN_FILE, answer_field=RESPONSE_FIELD)
# ...
```


## 11. 对应脚本命令

Notebook 会先把原始数据写成 `outputs/processed_train_messages.json` 和 `outputs/processed_test_messages.json`。Notebook 跑通后，正式训练可以直接用标准化后的 messages 文件：

```bash
CUDA_VISIBLE_DEVICES=1 python scripts/train_lora.py \
  --model_name_or_path model/Qwen3-1.7B \
  --train_file outputs/processed_train_messages.json \
  --output_dir outputs/qwen3_1p7b_unsloth_lora

CUDA_VISIBLE_DEVICES=1 python scripts/batch_infer_lora.py \
  --model_name_or_path outputs/qwen3_1p7b_unsloth_lora \
  --input_file outputs/processed_test_messages.json \
  --output_file outputs/qwen3_1p7b_predictions.json \
  --overwrite
```
